# Custom Object Detection with the TensorFlow Object Detection API

Workflow followed in this notebook:

1. Annotate images with **labelImg** (Pascal VOC XML)
2. Convert annotations to **TFRecords**
3. Fine-tune a pretrained **SSD MobileNet V2 FPNLite** from the TF2 Model Zoo (transfer learning)
4. Export the trained model as a `saved_model`
5. Run inference and look at the results

Training is slow on CPU, so a GPU (or Colab) is recommended for step 3.

In [ ]:
import random
import tarfile
import urllib.request
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from detect import predict
from utils import draw_detections, load_label_map, write_label_map

# ---- edit these for your own dataset ------------------------------
CLASSES = ["helmet", "no_helmet"]
MODEL_NAME = "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8"
NUM_STEPS = 3000
BATCH_SIZE = 4
# --------------------------------------------------------------------

ROOT = Path.cwd()
DATA = ROOT / "data"
TRAIN_DIR = DATA / "images" / "train"
TEST_DIR = DATA / "images" / "test"
LABEL_MAP = DATA / "label_map.pbtxt"
PRETRAINED = ROOT / "pretrained"
MY_MODEL = ROOT / "my_model"
EXPORT_DIR = ROOT / "exports"
TFOD = ROOT / "models" / "research" / "object_detection"

## 1. Dataset

Images are labelled with labelImg, which saves one XML file next to each image.
Split them roughly 80/20 into `data/images/train` and `data/images/test`.

In [ ]:
DATA.mkdir(exist_ok=True)
write_label_map(CLASSES, LABEL_MAP)
print(LABEL_MAP.read_text())

Before training, look at the labels. Most bad models come from bad annotations.

In [ ]:
def read_boxes(xml_file):
    root = ET.parse(xml_file).getroot()
    boxes = []
    for obj in root.findall("object"):
        b = obj.find("bndbox")
        coords = [int(float(b.find(k).text)) for k in ("xmin", "ymin", "xmax", "ymax")]
        boxes.append((obj.find("name").text, *coords))
    return boxes


def show_samples(folder, n=6):
    xmls = sorted(folder.glob("*.xml"))
    if not xmls:
        print(f"No annotations found in {folder}")
        return

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax in axes.flat:
        ax.axis("off")

    for ax, xml_file in zip(axes.flat, random.sample(xmls, min(n, len(xmls)))):
        image_file = next(p for p in folder.glob(xml_file.stem + ".*") if p.suffix.lower() != ".xml")
        img = cv2.cvtColor(cv2.imread(str(image_file)), cv2.COLOR_BGR2RGB)
        for name, x1, y1, x2, y2 in read_boxes(xml_file):
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(img, name, (x1, max(y1 - 8, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        ax.imshow(img)
        ax.set_title(image_file.name, fontsize=9)

    plt.tight_layout()
    plt.show()


def class_counts(folder):
    counts = Counter()
    for xml_file in folder.glob("*.xml"):
        counts.update(name for name, *_ in read_boxes(xml_file))
    return counts


show_samples(TRAIN_DIR)

train_counts, test_counts = class_counts(TRAIN_DIR), class_counts(TEST_DIR)
x, w = np.arange(len(CLASSES)), 0.38
plt.figure(figsize=(6, 3.5))
plt.bar(x - w / 2, [train_counts[c] for c in CLASSES], w, label="train")
plt.bar(x + w / 2, [test_counts[c] for c in CLASSES], w, label="test")
plt.xticks(x, CLASSES)
plt.ylabel("boxes")
plt.title("Annotations per class")
plt.legend()
plt.show()

## 2. Convert annotations to TFRecords

In [ ]:
!python generate_tfrecord.py -x {TRAIN_DIR} -l {LABEL_MAP} -o {DATA / "train.record"}
!python generate_tfrecord.py -x {TEST_DIR} -l {LABEL_MAP} -o {DATA / "test.record"}

## 3. Install the TensorFlow Object Detection API

Run once. This clones the TensorFlow `models` repo and builds the protobufs.

In [ ]:
!git clone --depth 1 https://github.com/tensorflow/models.git
%cd models/research
!protoc object_detection/protos/*.proto --python_out=.
!cp object_detection/packages/tf2/setup.py .
!pip install .
%cd ../..

## 4. Download the pretrained model

In [ ]:
URL = f"http://download.tensorflow.org/models/object_detection/tf2/20200711/{MODEL_NAME}.tar.gz"

PRETRAINED.mkdir(exist_ok=True)
archive = PRETRAINED / f"{MODEL_NAME}.tar.gz"
if not archive.exists():
    urllib.request.urlretrieve(URL, archive)

with tarfile.open(archive) as tar:
    tar.extractall(PRETRAINED)

## 5. Point the pipeline config at our data

Copy the model's `pipeline.config` and change the number of classes, batch size,
checkpoint and TFRecord paths.

In [ ]:
from google.protobuf import text_format
from object_detection.protos import pipeline_pb2

MY_MODEL.mkdir(exist_ok=True)
CONFIG = MY_MODEL / "pipeline.config"

pipeline = pipeline_pb2.TrainEvalPipelineConfig()
with tf.io.gfile.GFile(str(PRETRAINED / MODEL_NAME / "pipeline.config")) as f:
    text_format.Merge(f.read(), pipeline)

pipeline.model.ssd.num_classes = len(CLASSES)
pipeline.train_config.batch_size = BATCH_SIZE
pipeline.train_config.fine_tune_checkpoint = str(PRETRAINED / MODEL_NAME / "checkpoint" / "ckpt-0")
pipeline.train_config.fine_tune_checkpoint_type = "detection"

pipeline.train_input_reader.label_map_path = str(LABEL_MAP)
pipeline.train_input_reader.tf_record_input_reader.input_path[:] = [str(DATA / "train.record")]
pipeline.eval_input_reader[0].label_map_path = str(LABEL_MAP)
pipeline.eval_input_reader[0].tf_record_input_reader.input_path[:] = [str(DATA / "test.record")]

with tf.io.gfile.GFile(str(CONFIG), "wb") as f:
    f.write(text_format.MessageToString(pipeline))

## 6. Train

In [ ]:
!python {TFOD}/model_main_tf2.py --model_dir={MY_MODEL} --pipeline_config_path={CONFIG} --num_train_steps={NUM_STEPS}

Watch the loss in TensorBoard. It should trend down; if it flatlines early,
check the annotations and the learning rate.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {MY_MODEL}/train

Optional: evaluate the latest checkpoint on the test set (mAP shows up in TensorBoard).

In [ ]:
!python {TFOD}/model_main_tf2.py --model_dir={MY_MODEL} --pipeline_config_path={CONFIG} --checkpoint_dir={MY_MODEL}

## 7. Export the trained model

In [ ]:
!python {TFOD}/exporter_main_v2.py --input_type=image_tensor --pipeline_config_path={CONFIG} --trained_checkpoint_dir={MY_MODEL} --output_directory={EXPORT_DIR}

## 8. Inference on test images

In [ ]:
detect_fn = tf.saved_model.load(str(EXPORT_DIR / "saved_model"))
labels = load_label_map(LABEL_MAP)

test_images = sorted(p for p in TEST_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax in axes.flat:
    ax.axis("off")

for ax, path in zip(axes.flat, random.sample(test_images, min(6, len(test_images)))):
    frame = cv2.imread(str(path))
    boxes, scores, classes = predict(detect_fn, frame)
    shown = draw_detections(frame, boxes, scores, classes, labels, threshold=0.5)
    ax.imshow(cv2.cvtColor(shown, cv2.COLOR_BGR2RGB))

plt.tight_layout()
plt.show()

For real-time detection on a webcam or video, use the script instead:

```
python detect.py --source 0
```